In [3]:
import numpy as np
import pandas as pd

# -----------------------------
# 1. BASIC SIMULATION SETTINGS
# -----------------------------
np.random.seed(42)

DAYS = 30
PERIODS_PER_DAY = 96      # 15-minute resolution
H = DAYS * PERIODS_PER_DAY
DT = 0.25                 # hours per time slot (15 minutes)

NUM_CONSUMERS = 3
NUM_PROSUMERS = 7
N = NUM_CONSUMERS + NUM_PROSUMERS

# ---------------------------------------------------
# 2. HELPER: CREATE TIME INDEX & HOUSEHOLD LABELS
# ---------------------------------------------------
time_index = pd.date_range("2025-01-01", periods=H, freq="15min")

In [4]:
time_index

DatetimeIndex(['2025-01-01 00:00:00', '2025-01-01 00:15:00',
               '2025-01-01 00:30:00', '2025-01-01 00:45:00',
               '2025-01-01 01:00:00', '2025-01-01 01:15:00',
               '2025-01-01 01:30:00', '2025-01-01 01:45:00',
               '2025-01-01 02:00:00', '2025-01-01 02:15:00',
               ...
               '2025-01-30 21:30:00', '2025-01-30 21:45:00',
               '2025-01-30 22:00:00', '2025-01-30 22:15:00',
               '2025-01-30 22:30:00', '2025-01-30 22:45:00',
               '2025-01-30 23:00:00', '2025-01-30 23:15:00',
               '2025-01-30 23:30:00', '2025-01-30 23:45:00'],
              dtype='datetime64[ns]', length=2880, freq='15min')

In [5]:
household_ids = []
household_types = []   # "consumer" or "prosumer"

for i in range(NUM_CONSUMERS):
    household_ids.append(f"C{i+1}")
    household_types.append("consumer")

for j in range(NUM_PROSUMERS):
    household_ids.append(f"P{j+1}")
    household_types.append("prosumer")

household_ids = np.array(household_ids)
household_types = np.array(household_types)

In [6]:
household_ids, household_types

(array(['C1', 'C2', 'C3', 'P1', 'P2', 'P3', 'P4', 'P5', 'P6', 'P7'],
       dtype='<U2'),
 array(['consumer', 'consumer', 'consumer', 'prosumer', 'prosumer',
        'prosumer', 'prosumer', 'prosumer', 'prosumer', 'prosumer'],
       dtype='<U8'))

In [7]:
lambda_buy_individual = np.zeros(N)
lambda_sell_individual = np.zeros(N)

for i in range(N):
    if household_types[i] == "consumer":
        # Slightly higher / residential slab
        lambda_buy_individual[i] = np.random.uniform(6.0, 7.0)
        lambda_sell_individual[i] = 0.0   # no PV; export irrelevant
    else:
        # Prosumer: maybe special net-metering category
        lambda_buy_individual[i] = np.random.uniform(5.0, 6.0)
        lambda_sell_individual[i] = np.random.uniform(3.0, 3.8)

In [8]:
lambda_buy_individual

array([6.37454012, 6.95071431, 6.73199394, 5.59865848, 5.15599452,
       5.86617615, 5.70807258, 5.96990985, 5.21233911, 5.18340451])

In [9]:
lambda_sell_individual

array([0.        , 0.        , 0.        , 3.12481491, 3.04646689,
       3.48089201, 3.0164676 , 3.66595411, 3.14545997, 3.24339379])

In [15]:
def generate_daily_load_shape(periods_per_day=PERIODS_PER_DAY, dt=DT):
    """
    Create a generic 1-day load shape with morning & evening peaks.
    Returns an array of kW values (relative shape, not scaled to kWh).
    """
    t = np.arange(periods_per_day)
    print("--t--\n",t)
    hours = t * dt
    print("--hours--\n",hours)

    # Base: low at night, higher in evening
    base = 0.2
    morning_peak = np.exp(-0.5 * ((hours - 8) / 1.5) ** 2)
    print("--morning_peak--\n", morning_peak)
    evening_peak = 1.3 * np.exp(-0.5 * ((hours - 20) / 2.0) ** 2)
    print("--evening_peak--\n", evening_peak)

    shape = base + 0.7 * morning_peak + 1.0 * evening_peak
    shape = np.maximum(shape, 0.05)  # avoid zero
    return shape

In [17]:
generate_daily_load_shape(periods_per_day=PERIODS_PER_DAY, dt=DT), generate_daily_load_shape(periods_per_day=PERIODS_PER_DAY, dt=DT).shape

--t--
 [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23
 24 25 26 27 28 29 30 31 32 33 34 35 36 37 38 39 40 41 42 43 44 45 46 47
 48 49 50 51 52 53 54 55 56 57 58 59 60 61 62 63 64 65 66 67 68 69 70 71
 72 73 74 75 76 77 78 79 80 81 82 83 84 85 86 87 88 89 90 91 92 93 94 95]
--hours--
 [ 0.    0.25  0.5   0.75  1.    1.25  1.5   1.75  2.    2.25  2.5   2.75
  3.    3.25  3.5   3.75  4.    4.25  4.5   4.75  5.    5.25  5.5   5.75
  6.    6.25  6.5   6.75  7.    7.25  7.5   7.75  8.    8.25  8.5   8.75
  9.    9.25  9.5   9.75 10.   10.25 10.5  10.75 11.   11.25 11.5  11.75
 12.   12.25 12.5  12.75 13.   13.25 13.5  13.75 14.   14.25 14.5  14.75
 15.   15.25 15.5  15.75 16.   16.25 16.5  16.75 17.   17.25 17.5  17.75
 18.   18.25 18.5  18.75 19.   19.25 19.5  19.75 20.   20.25 20.5  20.75
 21.   21.25 21.5  21.75 22.   22.25 22.5  22.75 23.   23.25 23.5  23.75]
--morning_peak--
 [6.65836147e-07 1.59725788e-06 3.72665317e-06 8.45666597e-06
 1.86644691e-05 4.0065297

(array([0.20000047, 0.20000112, 0.20000261, 0.20000592, 0.20001307,
        0.20002805, 0.20005855, 0.2001189 , 0.20023482, 0.20045107,
        0.2008427 , 0.20153124, 0.20270614, 0.20465151, 0.2077763 ,
        0.21264411, 0.21999585, 0.23075585, 0.24600997, 0.26694411,
        0.2947347 , 0.33038932, 0.37454655, 0.42725673, 0.4877786 ,
        0.55443493, 0.62457146, 0.69465379, 0.76051618, 0.81774783,
        0.86217163, 0.89034499, 0.90000002, 0.89034502, 0.86217171,
        0.81774801, 0.76051653, 0.69465449, 0.62457281, 0.5544375 ,
        0.48778345, 0.42726571, 0.37456294, 0.33041877, 0.29478678,
        0.26703481, 0.24616547, 0.2310183 , 0.22043195, 0.21335752,
        0.20892527, 0.20647328, 0.20554988, 0.20590144, 0.20745459,
        0.21029945, 0.21467652, 0.22096792, 0.22969099, 0.24149128,
        0.25713108, 0.27746943, 0.30342997, 0.33595382, 0.37593633,
        0.4241483 , 0.48114479, 0.54716741, 0.62204822, 0.70512557,
        0.79518337, 0.89042479, 0.98848986, 1.08